# Amazon Delivery Time Prediction - Notebook

In [ ]:
# ================================
# 1. Imports
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [ ]:
# ================================
# 2. Load Dataset
# ================================
df = pd.read_csv("amazon_delivery.csv")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# ================================
# 3. Feature Engineering
# ================================
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # km

df['Order_DateTime'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Order_Time'], errors='coerce')
df['Pickup_DateTime'] = pd.to_datetime(df['Pickup_Time'], errors='coerce')
df['Pickup_Lag_min'] = (df['Pickup_DateTime'] - df['Order_DateTime']).dt.total_seconds() / 60.0

df['Distance_km'] = haversine_distance(
    df['Store_Latitude'], df['Store_Longitude'],
    df['Drop_Latitude'], df['Drop_Longitude']
)

df['Hour'] = df['Order_DateTime'].dt.hour
df['DayOfWeek'] = df['Order_DateTime'].dt.dayofweek

for col in ['Weather','Traffic','Vehicle','Area','Category']:
    if col in df.columns:
        df[col] = df[col].astype('category').cat.codes

df['Delivery_Time'] = pd.to_numeric(df['Delivery_Time'], errors='coerce')
df = df.dropna(subset=['Delivery_Time'])

print("Processed dataset shape:", df.shape)
df.head()

In [ ]:
# ================================
# 4. EDA
# ================================
plt.figure(figsize=(6,4))
sns.histplot(df['Delivery_Time'], bins=30, kde=True)
plt.title("Delivery Time Distribution")
plt.show()

plt.figure(figsize=(6,4))
sns.heatmap(df.corr(), annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ================================
# 5. Train-Test Split
# ================================
X = df.drop(columns=['Delivery_Time','Order_ID','Order_Date','Order_Time','Pickup_Time','Order_DateTime','Pickup_DateTime'])
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ================================
# 6. Train Models
# ================================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, random_state=42, learning_rate=0.1)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    print(f"{name} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}")

In [ ]:
# ================================
# 7. Select Best Model
# ================================
best_model_name = min(results, key=lambda k: results[k]['RMSE'])
best_model = models[best_model_name]
print("Best Model:", best_model_name, "with RMSE:", results[best_model_name]['RMSE'])

joblib.dump(best_model, "best_model.pkl")

In [ ]:
# ================================
# 8. Prediction Demo
# ================================
sample = X_test.iloc[[0]]
print("Sample input:\n", sample)
pred = best_model.predict(sample)[0]
print("Predicted Delivery Time:", pred, "hours")
print("Actual Delivery Time:", y_test.iloc[0], "hours")